# CBAG — AccessPoint/UAV2 (targeted-to-Normal), phased structure — SAME code as UAV1
AccessPoint has a real Normal class (20.9%), so the standard targeted CBAG (P(Normal)>=0.5) works directly. Set the AccessPoint `glob` in CELL 1, run cells top-to-bottom.


In [2]:
# ===== CELL 1 — CONFIG + imports + cache helpers =====
import os,glob,gc,json,pickle,warnings,time,numpy as np,pandas as pd
warnings.filterwarnings("ignore")
try:
    import torch, torch.nn as nn; HAVE_TORCH=True; torch.set_num_threads(os.cpu_count() or 4)
except Exception:
    HAVE_TORCH=False
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from scipy.stats import wilcoxon
def P(*a): print(*a,flush=True)
SEED=42; np.random.seed(SEED)
if HAVE_TORCH: torch.manual_seed(SEED)

# ---------------- CONTROL PANEL ----------------
DATASET   = "AccessPoint"          # "UAV1" | "AccessPoint" | "UAV3_GSC"  <-- confirmed working, real Normal class 20.9%
SUBSAMPLE = False          # FULL data (chunked loader below keeps RAM low)
PER_CLASS_SUB = 30000
TARGET    = "cnn"           # "cnn" (CNN-1D, needs torch) else auto MLP fallback
CNN_PER_CLASS = 5500; CNN_EPOCHS = 6
GENERATORS = ["C-SA","C-RL","C-PSO","C-GA","C-BO","CG-Diffusion","CC-CGAN","HopSkipJump","Sign-OPT"]  # FULL proposal roster
ENABLE_HEAVY = True    # run heavy methods (C-BO/CG-Diffusion/CC-CGAN/HopSkipJump/Sign-OPT); False -> skip (BO/Diff use projected)
EVAL_CAP  = 400             # detected samples/class fed to each generator
# ---- Tri-Gate (targeted) ----
# EXTRACT survivors at ANY gate config you want (no regeneration):
# phase3(theta_mode="fixed", theta_fix=0.5, gate2_cmr=False, gate3=0.5, save_name="survivors_L1_0.5")

GATE1_THR = 0.5            # Gate-1 evasion: P(Normal) >= this
GATE2_CMR = False           # Gate-2 = oracle only (True = oracle & CMR, func2 original)
THETA_MODE= "fixed"         # "fixed" -> THETA_FIX ; "paper" -> max(0.5, p10)
THETA_FIX = 0.50
GATE3_OP  = 0.10           # Gate-3 operating point (stealthy)
G3_SWEEP  = [0.10,0.25,0.50,1.00,2.00]
HARD_SD_K = 3.0; CMR_PCT = 90
IMMUTABLE={'ip.proto','ip.ttl','ip.dst','id.orig_h','id.resp_h'}
UNSETTABLE={'radiotap.dbm_antsignal','wlan_radio.signal_dbm','radiotap.datarate',
 'radiotap.channel.freq','wlan_radio.frequency','radiotap.channel.flags.ofdm'}
LEAKAGE=['frame.number','frame.time_relative','frame.time_epoch','radiotap.timestamp.ts','radiotap.mactime',
 'wlan_radio.start_tsf','wlan_radio.end_tsf','wlan.seq','radiotap.length','uid','id.resp_h','id.orig_p','id.resp_p']
DATASETS={
 "UAV1":       dict(glob="/kaggle/input/datasets/saleemsarg/cbag-uav/UAV-Case1-Label (1).csv", label="Label", benign="Normal",
    drop=["Reconnassiance","Reconnaissance"],
    IG=['radiotap.channel.flags.cck','frame.encap_type','radiotap.channel.freq','ip.proto','ip.ttl','wlan.fixed.capabilities.ess','ip.dst','frame.len','udp.dstport','wlan.fc.subtype'],
    LIME=['radiotap.channel.flags.cck','wlan.fixed.beacon','arp','radiotap.channel.flags.ofdm','wlan.fc.subtype','tcp.ack','wlan.fc.retry','frame.time_delta_displayed','wlan.bssid','udp.length'],
    craft=['wlan.fc.retry','wlan.fcs.bad_checksum','wlan.rsn.capabilities.mfpc','frame.encap_type']),
 "AccessPoint":dict(glob="/kaggle/input/datasets/saleemsarg/cbag-uav/Access Point Case2 Label.csv", label="Normal", benign="Normal",
    drop=["Reconnaissance","Ewil Twin"],
    IG=['radiotap.rxflags','wlan.fcs.bad_checksum','ip.ttl','ip.proto','udp.dstport','frame.len','radiotap.channel.flags.cck','wlan.fc.subtype','wlan.fc.retry','udp.length'],
    LIME=['radiotap.rxflags','wlan.fc.pwrmgt','wlan.fc.retry','udp.length','nbss.continuation_data','wlan.fc.subtype','radiotap.channel.flags.cck','nbns','radiotap.present.tsft','wlan_radio.data_rate'],
    craft=['wlan.fc.retry','wlan.fcs.bad_checksum','wlan.fc.pwrmgt','frame.encap_type']),
 "UAV3_GSC":   dict(glob="/kaggle/input/datasets/saleemsarg/cbag-uav/GSC Case3 Label .csv", label="Class", benign="Benign",
    drop=["Evil Twin"],
    IG=['fwd_header_size_min','idle.std','bwd_header_size_tot','bwd_header_size_min','fwd_header_size_tot','bwd_header_size_max','bwd_pkts_per_sec','bwd_init_window_size','bwd_data_pkts_tot','id.orig_h'],
    LIME=['fwd_header_size_min','fwd_PSH_flag_count','fwd_pkts_payload.avg','fwd_pkts_payload.max','bwd_header_size_tot','flow_ECE_flag_count','bwd_pkts_payload.avg','bwd_pkts_payload.tot','fwd_pkts_payload.min','bwd_pkts_payload.std'],
    craft=[]),
}
cfg=DATASETS[DATASET]; BEN=cfg["benign"]; craft=cfg["craft"]
CACHE=f"cbag_cache_{DATASET}"; OUT=f"cbag_out_{DATASET}"; os.makedirs(CACHE,exist_ok=True); os.makedirs(OUT,exist_ok=True)
def feature_pool(feat,difs): return [f for f in difs if f not in UNSETTABLE]
def has(t): return os.path.exists(f"{CACHE}/{t}.pkl")
def save(t,o): pickle.dump(o,open(f"{CACHE}/{t}.pkl","wb"))
def load(t): return pickle.load(open(f"{CACHE}/{t}.pkl","rb"))
P(f"CONFIG ok. DATASET={DATASET} SUBSAMPLE={SUBSAMPLE} TARGET={TARGET} torch={HAVE_TORCH} GENERATORS={GENERATORS}")


CONFIG ok. DATASET=AccessPoint SUBSAMPLE=True TARGET=cnn torch=True GENERATORS=['C-SA', 'C-RL', 'C-PSO', 'C-GA', 'C-BO', 'CG-Diffusion', 'CC-CGAN', 'HopSkipJump', 'Sign-OPT']


In [3]:
# ===== CELL 2 — REAL generators (inlined; no external file) =====
def C_SA(det,lo,hi,mut,pN,L1,rng,wp=0.3,iters=400,cool=0.98,step=0.5,track=False):
    Xa=det.copy()
    for j in mut: Xa[:,j]=np.clip(Xa[:,j],lo[j],hi[j])
    en=lambda Z:-pN(Z)+wp*L1(Z,det,mut); E=en(Xa); T=1.0; curve=[]
    for it in range(iters):
        c=Xa.copy(); jj=np.array(mut)[rng.integers(0,len(mut),len(Xa))]
        c[np.arange(len(Xa)),jj]+=rng.normal(0,step,len(Xa)).astype(np.float32)
        for j in mut: c[:,j]=np.clip(c[:,j],lo[j],hi[j])
        Ec=en(c); dE=Ec-E; acc=(dE<0)|(rng.random(len(Xa))<np.exp(-np.clip(dE,0,50)/max(T,1e-3)))
        Xa[acc]=c[acc]; E[acc]=Ec[acc]; T*=cool
        if track and (it%20==0 or it==iters-1): curve.append([it,float((pN(Xa)>=GATE1_THR).mean()*100)])
    return (Xa,iters,curve) if track else (Xa,iters)

def C_PSO(det,lo,hi,mut,pN,L1,rng,wp=0.3,Pn=12,iters=40):
    N,d,F=len(det),len(mut),det.shape[1]; base=np.repeat(det[:,None,:],Pn,1).astype(np.float32)
    base[:,:,mut]=rng.uniform(lo[mut],hi[mut],(N,Pn,d)).astype(np.float32); vel=np.zeros((N,Pn,d),np.float32)
    en=lambda B:-pN(B.reshape(-1,F)).reshape(N,Pn)+wp*(np.abs(B[:,:,mut]-det[:,None,mut])/(hi[mut]-lo[mut]+1e-9)).sum(2)
    E=en(base); pb=base.copy(); pbE=E.copy(); gi=pbE.argmin(1); gb=pb[np.arange(N),gi].copy()
    for _ in range(iters):
        r1=rng.random((N,Pn,d)).astype(np.float32); r2=rng.random((N,Pn,d)).astype(np.float32)
        vel=0.6*vel+1.5*r1*(pb[:,:,mut]-base[:,:,mut])+1.5*r2*(gb[:,None,mut]-base[:,:,mut])
        base[:,:,mut]=np.clip(base[:,:,mut]+vel,lo[mut],hi[mut]); E=en(base)
        imp=E<pbE; pb[imp]=base[imp]; pbE[imp]=E[imp]; gi=pbE.argmin(1); gb=pb[np.arange(N),gi].copy()
    return gb, Pn*iters

def C_GA(det,lo,hi,mut,pN,L1,rng,wp=0.3,Pn=12,gens=40):
    N,d,F=len(det),len(mut),det.shape[1]; pop=np.repeat(det[:,None,:],Pn,1).astype(np.float32)
    pop[:,:,mut]=rng.uniform(lo[mut],hi[mut],(N,Pn,d)).astype(np.float32)
    fit=lambda B:-(-pN(B.reshape(-1,F)).reshape(N,Pn)+wp*(np.abs(B[:,:,mut]-det[:,None,mut])/(hi[mut]-lo[mut]+1e-9)).sum(2))
    for _ in range(gens):
        f=fit(pop); order=np.argsort(-f,1); elite=np.take_along_axis(pop,order[:,:Pn//2,None],1); off=elite.copy()
        msk=rng.random((N,Pn//2,d))<0.3
        off[:,:,mut]=np.clip(off[:,:,mut]+msk*rng.normal(0,0.3,(N,Pn//2,d)).astype(np.float32),lo[mut],hi[mut])
        pop=np.concatenate([elite,off],1)
    f=fit(pop); gi=f.argmax(1); return pop[np.arange(N),gi], Pn*gens

def C_RL(det,tr,lo,hi,mut,pN,L1,rng,wp=0.3,T=6,iters=40,step=0.35):
    if not HAVE_TORCH: return C_SA(det,lo,hi,mut,pN,L1,rng,wp=wp)   # SA fallback if no torch
    d=len(mut)
    class Pol(nn.Module):
        def __init__(s): super().__init__(); s.n=nn.Sequential(nn.Linear(d,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh()); s.mu=nn.Linear(64,d); s.ls=nn.Parameter(torch.full((d,),-0.5))
        def forward(s,x): h=s.n(x); return s.mu(h),s.ls.exp()
    pol=Pol(); opt=torch.optim.Adam(pol.parameters(),3e-3); loj,hij=lo[mut],hi[mut]
    for _ in range(iters):
        b=tr[rng.choice(len(tr),min(256,len(tr)),replace=False)].copy(); x=b.copy(); lps=[]
        for _ in range(T):
            mu,sd=pol(torch.tensor(x[:,mut])); dist=torch.distributions.Normal(mu,sd); a=dist.sample()
            lps.append(dist.log_prob(a).sum(1)); x[:,mut]=np.clip(x[:,mut]+step*a.numpy(),loj,hij)
        R=torch.tensor(pN(x)-wp*L1(x,b,mut),dtype=torch.float32); adv=R-R.mean()
        loss=-(torch.stack(lps).sum(0)*adv).mean(); opt.zero_grad(); loss.backward(); opt.step()
    x=det.copy()
    for _ in range(T):
        with torch.no_grad(): mu,sd=pol(torch.tensor(x[:,mut]))
        x[:,mut]=np.clip(x[:,mut]+step*mu.numpy(),loj,hij)
    return x, iters*T

HSJ_MAX=60; BO_MAX=60   # per-class sample cap for the slow per-sample methods (raise for more samples on GPU)
def C_BO(det,lo,hi,mut,pN,L1,rng,wp=0.3,init=8,iters=20):
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import Matern
    from scipy.stats import norm as _norm
    d=len(mut); loj=lo[mut]; hij=hi[mut]; out=det.copy()
    for idx in range(min(len(det),BO_MAX)):
        x0=det[idx]
        def obj(delta):
            c=x0.copy(); c[mut]=np.clip(delta,loj,hij); z=c[None,:]; return float(-pN(z)[0]+wp*L1(z,x0[None,:],mut)[0])
        Xs=rng.uniform(loj,hij,(init,d)); Ys=np.array([obj(v) for v in Xs])
        for _ in range(iters):
            gp=GaussianProcessRegressor(kernel=Matern(nu=2.5),normalize_y=True,alpha=1e-4,n_restarts_optimizer=0).fit(Xs,Ys)
            cand=rng.uniform(loj,hij,(256,d)); mu,sd=gp.predict(cand,return_std=True)
            imp=Ys.min()-mu; Z=imp/(sd+1e-9); ei=imp*_norm.cdf(Z)+sd*_norm.pdf(Z)
            nx=cand[ei.argmax()]; Xs=np.vstack([Xs,nx]); Ys=np.append(Ys,obj(nx))
        c=x0.copy(); c[mut]=np.clip(Xs[Ys.argmin()],loj,hij); out[idx]=c
    return out, init+iters

def CG_Diffusion(det,tr,lo,hi,mut,pN,L1,rng,wp=0.3,steps=16,epochs=40,guide=15):
    if not HAVE_TORCH: return C_SA(det,lo,hi,mut,pN,L1,rng,wp=wp)
    d=len(mut); w=(hi[mut]-lo[mut]+1e-9); nrm=lambda a:(a-lo[mut])/w; dn=lambda a:a*w+lo[mut]
    Tr=torch.tensor(np.clip(nrm(tr[:,mut]),0,1).astype(np.float32))
    betas=torch.linspace(1e-4,0.05,steps); ab=torch.cumprod(1-betas,0)
    net=nn.Sequential(nn.Linear(d+1,128),nn.SiLU(),nn.Linear(128,128),nn.SiLU(),nn.Linear(128,d)); opt=torch.optim.Adam(net.parameters(),2e-3)
    for _ in range(epochs):
        i=torch.randint(0,len(Tr),(min(256,len(Tr)),)); x0=Tr[i]; t=torch.randint(0,steps,(len(x0),)); a=ab[t][:,None]
        nz=torch.randn_like(x0); xt=torch.sqrt(a)*x0+torch.sqrt(1-a)*nz
        loss=((net(torch.cat([xt,t[:,None].float()/steps],1))-nz)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
    x=torch.tensor(np.clip(nrm(det[:,mut]),0,1).astype(np.float32)); x=torch.sqrt(ab[-1])*x+torch.sqrt(1-ab[-1])*torch.randn_like(x)
    with torch.no_grad():
        for t in reversed(range(steps)):
            tt=torch.full((len(x),),t); eps=net(torch.cat([x,tt[:,None].float()/steps],1)); a=ab[t]
            x0h=torch.clamp((x-torch.sqrt(1-a)*eps)/torch.sqrt(a),0,1)
            x=x0h if t==0 else torch.sqrt(ab[t-1])*x0h+torch.sqrt(1-ab[t-1])*torch.randn_like(x)
    cur=det.copy(); cur[:,mut]=np.clip(dn(x.numpy()),lo[mut],hi[mut]); E=-pN(cur)+wp*L1(cur,det,mut)
    for _ in range(guide):
        c=cur.copy(); jj=np.array(mut)[rng.integers(0,len(mut),len(cur))]; c[np.arange(len(cur)),jj]+=rng.normal(0,0.4,len(cur)).astype(np.float32)
        for j in mut: c[:,j]=np.clip(c[:,j],lo[j],hi[j])
        Ec=-pN(c)+wp*L1(c,det,mut); acc=Ec<E; cur[acc]=c[acc]; E[acc]=Ec[acc]
    return cur, steps*epochs+guide

def CC_CGAN(det,tr,lo,hi,mut,pN,L1,rng,wp=0.3,epochs=60,zdim=8,guide=15):
    if not HAVE_TORCH: return C_GA(det,lo,hi,mut,pN,L1,rng,wp=wp)
    d=len(mut); w=(hi[mut]-lo[mut]+1e-9); nrm=lambda a:(a-lo[mut])/w; dn=lambda a:a*w+lo[mut]
    R=torch.tensor(np.clip(nrm(tr[:,mut]),0,1).astype(np.float32))
    G=nn.Sequential(nn.Linear(zdim,64),nn.ReLU(),nn.Linear(64,d),nn.Sigmoid()); D=nn.Sequential(nn.Linear(d,64),nn.LeakyReLU(0.2),nn.Linear(64,1))
    og=torch.optim.Adam(G.parameters(),2e-3); od=torch.optim.Adam(D.parameters(),2e-3); bce=nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        idx=torch.randint(0,len(R),(min(256,len(R)),)); real=R[idx]; z=torch.randn(len(real),zdim); fake=G(z).detach()
        od.zero_grad(); (bce(D(real),torch.ones(len(real),1))+bce(D(fake),torch.zeros(len(real),1))).backward(); od.step()
        og.zero_grad(); z=torch.randn(len(real),zdim); bce(D(G(z)),torch.ones(len(real),1)).backward(); og.step()
    with torch.no_grad(): gen=G(torch.randn(len(det),zdim)).numpy()
    cur=det.copy(); cur[:,mut]=np.clip(dn(gen),lo[mut],hi[mut]); E=-pN(cur)+wp*L1(cur,det,mut)
    for _ in range(guide):
        c=cur.copy(); jj=np.array(mut)[rng.integers(0,len(mut),len(cur))]; c[np.arange(len(cur)),jj]+=rng.normal(0,0.4,len(cur)).astype(np.float32)
        for j in mut: c[:,j]=np.clip(c[:,j],lo[j],hi[j])
        Ec=-pN(c)+wp*L1(c,det,mut); acc=Ec<E; cur[acc]=c[acc]; E[acc]=Ec[acc]
    return cur, epochs+guide

def _rand_unc(det,lo,hi,mut,pN,L1,rng,**k):
    x=det.copy(); x[:,mut]=rng.uniform(lo[mut],hi[mut],(len(det),len(mut))).astype(np.float32); return x,1

def HopSkipJump(det,lo,hi,mut,pN,L1,rng,ref=None,iters=12,B=18,**k):
    N=min(len(det),HSJ_MAX); out=det.copy(); ref=det.mean(0) if ref is None else ref
    for i in range(N):
        orig=det[i]; adv=ref.copy()
        if pN(adv[None])[0]<0.5: out[i]=orig; continue
        for _ in range(iters):
            a,b=0.0,1.0
            for _ in range(8):
                m=(a+b)/2; c=orig.copy(); c[mut]=orig[mut]+m*(adv[mut]-orig[mut]); (a,b)=((a,m) if pN(c[None])[0]>=0.5 else (m,b))
            bnd=orig.copy(); bnd[mut]=orig[mut]+b*(adv[mut]-orig[mut])
            U=rng.normal(0,1,(B,len(mut))); U/=np.linalg.norm(U,axis=1,keepdims=True)+1e-9
            cand=np.repeat(bnd[None],B,0).astype(np.float32); cand[:,mut]=np.clip(cand[:,mut]+0.05*U,lo[mut],hi[mut])
            phi=(pN(cand)>=0.5).astype(float)*2-1; grad=(phi[:,None]*U).mean(0); adv=bnd.copy(); adv[mut]=np.clip(bnd[mut]+0.1*grad,lo[mut],hi[mut])
        out[i]=adv
    return out, iters*(8+B)

def Sign_OPT(det,lo,hi,mut,pN,L1,rng,ref=None,iters=12,Q=20,**k):
    N=min(len(det),HSJ_MAX); out=det.copy(); ref=det.mean(0) if ref is None else ref
    for i in range(N):
        orig=det[i]; th=(ref[mut]-orig[mut]); th/=(np.linalg.norm(th)+1e-9)
        def g(t):
            a,b=0.0,4.0
            for _ in range(10):
                m=(a+b)/2; c=orig.copy(); c[mut]=np.clip(orig[mut]+m*t,lo[mut],hi[mut]); (a,b)=((a,m) if pN(c[None])[0]>=0.5 else (m,b))
            return b
        g0=g(th)
        for _ in range(iters):
            u=rng.normal(0,1,len(mut)); u/=np.linalg.norm(u)+1e-9; s=1 if g(th+0.1*u)<g0 else -1
            th=th+0.05*s*u; th/=np.linalg.norm(th)+1e-9; g0=g(th)
        c=orig.copy(); c[mut]=np.clip(orig[mut]+g0*th,lo[mut],hi[mut]); out[i]=c
    return out, iters*Q

GENS={"C-SA":C_SA,"C-RL":C_RL,"C-PSO":C_PSO,"C-GA":C_GA,"C-BO":C_BO,"CG-Diffusion":CG_Diffusion,"CC-CGAN":CC_CGAN,"HopSkipJump":HopSkipJump,"Sign-OPT":Sign_OPT}
NEEDS_TR={"C-RL","CG-Diffusion","CC-CGAN"}
HEAVY={"C-BO","CG-Diffusion","CC-CGAN","HopSkipJump","Sign-OPT"}
P("generators inlined (full proposal roster):",list(GENS))


generators inlined (full proposal roster): ['C-SA', 'C-RL', 'C-PSO', 'C-GA', 'C-BO', 'CG-Diffusion', 'CC-CGAN', 'HopSkipJump', 'Sign-OPT']


In [4]:
# ===== CELL 3 — PHASE 0: load & clean (CHUNKED / low-RAM, FULL data, no sampling) =====
def _hexnum(s):
    num=pd.to_numeric(s,errors="coerce"); m=num.isna()&s.notna()
    if m.any(): num.loc[m]=s[m].map(lambda v: float(int(str(v).strip(),16)) if str(v).strip().lower().startswith("0x") else np.nan)
    return num
def phase0(force=False):
    if has("p0") and not force: P("[P0] cached"); return load("p0")
    files=sorted(glob.glob(cfg["glob"])); assert files,"NO FILES at "+cfg["glob"]
    # PASS 1 (sample only): detect string/categorical columns for consistent factorization
    samp=pd.read_csv(files[0],low_memory=False,nrows=200000).rename(columns={cfg["label"]:"Label"}).drop(columns=["Label"])
    cols=list(samp.columns)
    catcols=set(c for c in cols if (_hexnum(samp[c]).isna()&samp[c].notna()).any())
    catmap={c:{} for c in catcols}; del samp; gc.collect()
    # PASS 2 (chunked): convert each chunk to float32 immediately, free it -> low peak RAM
    labs=[]; parts=[]; CH=150000
    for f in files:
        for d in pd.read_csv(f,low_memory=False,chunksize=CH):
            d=d.rename(columns={cfg["label"]:"Label"}); lab=d["Label"].astype(str); d=d.drop(columns=["Label"])
            keep=~lab.isin(cfg["drop"]); d=d[keep]; lab=lab[keep]; d=d.reindex(columns=cols)
            arr=np.empty((len(d),len(cols)),np.float32)
            for jc,c_ in enumerate(cols):
                if c_ in catcols:
                    v=d[c_].astype(str); cm=catmap[c_]
                    for u in v.unique():
                        if u not in cm: cm[u]=len(cm)
                    arr[:,jc]=v.map(cm).astype(np.float32).values
                else:
                    arr[:,jc]=_hexnum(d[c_]).fillna(0.0).astype(np.float32).values
            parts.append(arr); labs+=lab.tolist(); del d,arr; gc.collect()
    X=np.concatenate(parts,0); del parts; gc.collect(); X[~np.isfinite(X)]=0.0
    lo=np.quantile(X,0.0001,0); hi=np.quantile(X,0.9999,0); np.clip(X,lo,hi,out=X)     # in-place clip (no copy)
    nun=(X!=X[0]).any(0); drop=set([cols[j] for j in range(len(cols)) if not nun[j]])|set(LEAKAGE)
    kj=[j for j,c_ in enumerate(cols) if c_ not in drop]; X=np.ascontiguousarray(X[:,kj]); feat=[cols[j] for j in kj]
    difs=[d for d in list(dict.fromkeys(cfg["IG"]+cfg["LIME"])) if d in feat]; labs=np.array(labs)
    if SUBSAMPLE:
        rng=np.random.default_rng(SEED); idx=[]
        for c_ in np.unique(labs):
            ii=np.where(labs==c_)[0]; idx+=list(rng.choice(ii,min(PER_CLASS_SUB,len(ii)),replace=False))
        idx=np.array(idx); rng.shuffle(idx); X=X[idx]; labs=labs[idx]
    o=dict(X=X,labels=labs,feat=feat,difs=difs); save("p0",o); gc.collect()
    P(f"[P0] clean {X.shape} classes {len(np.unique(labs))} DIFs {len(difs)} {'(SUBSAMPLE)' if SUBSAMPLE else '(FULL, chunked low-RAM)'}")
    return o


In [5]:
# ===== CELL 4 — PHASE 1: target (CNN-1D/MLP) + oracle + CMR (cached) =====
if HAVE_TORCH:
    class CNN1D(nn.Module):
        def __init__(s,d,k):
            super().__init__()
            s.c=nn.Sequential(nn.Conv1d(1,64,3,padding=1),nn.BatchNorm1d(64),nn.ReLU(),nn.MaxPool1d(2),
                              nn.Conv1d(64,128,3,padding=1),nn.BatchNorm1d(128),nn.ReLU(),nn.AdaptiveMaxPool1d(1))
            s.f=nn.Sequential(nn.Flatten(),nn.Linear(128,128),nn.ReLU(),nn.Dropout(0.3),nn.Linear(128,k))
        def forward(s,x): return s.f(s.c(x.unsqueeze(1)))
def phase1(force=False):
    if has("p1") and not force: P("[P1] cached"); return load("p1")
    d=phase0(); X,labels,feat,difs=d["X"],d["labels"],d["feat"],d["difs"]
    mean_=X.mean(0); scale_=X.std(0); scale_[scale_<1e-8]=1; Xs=((X-mean_)/scale_).astype(np.float32)
    del X; gc.collect()
    le=LabelEncoder().fit(labels); y=le.transform(labels); CLS=list(le.classes_); NORMAL=le.transform([BEN])[0]
    Xtr,Xte,ytr,yte=train_test_split(Xs,y,test_size=0.25,stratify=y,random_state=SEED)
    kind="mlp"
    if TARGET=="cnn" and HAVE_TORCH:
        kind="cnn"; rng=np.random.default_rng(SEED); idx=[]
        for c in range(len(CLS)):
            ii=np.where(ytr==c)[0]; idx+=list(rng.choice(ii,min(CNN_PER_CLASS,len(ii)),replace=len(ii)<CNN_PER_CLASS))
        idx=np.array(idx); rng.shuffle(idx); Xt=torch.tensor(Xtr[idx]); yt=torch.tensor(ytr[idx]); m=CNN1D(len(feat),len(CLS))
        opt=torch.optim.Adam(m.parameters(),1e-3); ce=nn.CrossEntropyLoss(); m.train()
        for ep in range(CNN_EPOCHS):
            pm=torch.randperm(len(Xt))
            for i in range(0,len(Xt),256):
                b=pm[i:i+256]; opt.zero_grad(); ce(m(Xt[b]),yt[b]).backward(); opt.step()
        m.eval(); torch.save(m.state_dict(),f"{CACHE}/cnn.pt")
        def proba(Z):
            if len(Z)==0: return np.zeros((0,len(CLS)),np.float32)
            with torch.no_grad(): return np.concatenate([torch.softmax(m(torch.tensor(np.ascontiguousarray(Z[i:i+8192]))),1).numpy() for i in range(0,len(Z),8192)])
    else:
        m=MLPClassifier(hidden_layer_sizes=(64,),max_iter=60,alpha=1e-2,random_state=SEED).fit(Xtr,ytr); cl=list(m.classes_)
        pickle.dump(m,open(f"{CACHE}/mlp.pkl","wb"))
        def proba(Z):
            Pr=m.predict_proba(Z.astype(np.float32)); full=np.zeros((len(Z),len(CLS)),np.float32)
            for j,c in enumerate(cl): full[:,c]=Pr[:,j]
            return full
    acc=float((proba(Xte).argmax(1)==yte).mean())
    from sklearn.metrics import f1_score; f1=f1_score(yte,proba(Xte).argmax(1),average="macro")
    P(f"[P1] target={kind} acc={acc:.4f} macroF1={f1:.4f}")
    si=np.random.default_rng(1).choice(len(Xtr),min(30000,len(Xtr)),replace=False)
    oracle=HistGradientBoostingClassifier(max_iter=150,random_state=1).fit(Xtr[si],ytr[si])
    fidx={f:feat.index(f) for f in feat}; cmr={}
    for c in [x for x in CLS if x!=BEN]:
        ci=le.transform([c])[0]; Xc=Xtr[ytr==ci]
        if len(Xc)<60: continue
        pool=[f for f in feature_pool(feat,difs) if f not in IMMUTABLE and Xc[:,fidx[f]].std()>=0.05]
        if not pool: continue
        pidx=[fidx[f] for f in pool]; Xcs=Xc[np.random.default_rng(0).choice(len(Xc),min(20000,len(Xc)),replace=False)]
        km=KMeans(n_clusters=1,n_init=3,random_state=SEED).fit(Xcs[:,pidx]); dd=cdist(Xcs[:,pidx],km.cluster_centers_).min(1)
        cmr[c]=dict(pidx=pidx,centroids=km.cluster_centers_,eps=float(np.percentile(dd,CMR_PCT)))
    gmin,gmax=Xs.min(0),Xs.max(0); grng=np.where((gmax-gmin)>1e-9,gmax-gmin,1).astype(np.float32)
    o=dict(kind=kind,oracle=oracle,cmr=cmr,Xtr=Xtr,Xte=Xte,ytr=ytr,yte=yte,le=le,CLS=CLS,NORMAL=NORMAL,
           feat=feat,difs=difs,mean_=mean_,scale_=scale_,gmin=gmin,gmax=gmax,grng=grng,fidx=fidx,acc=acc,f1=f1,
           ncls=len(CLS),nfeat=len(feat)); save("p1",o); P("[P1] cached"); return o

def helpers(S):
    NORMAL=S["NORMAL"]; grng=S["grng"]
    if S["kind"]=="cnn":
        m=CNN1D(S["nfeat"],S["ncls"]); m.load_state_dict(torch.load(f"{CACHE}/cnn.pt")); m.eval()
        def proba(Z):
            if len(Z)==0: return np.zeros((0,S["ncls"]),np.float32)
            with torch.no_grad(): return np.concatenate([torch.softmax(m(torch.tensor(np.ascontiguousarray(Z[i:i+8192]))),1).numpy() for i in range(0,len(Z),8192)])
    else:
        m=pickle.load(open(f"{CACHE}/mlp.pkl","rb")); cl=list(m.classes_)
        def proba(Z):
            Pr=m.predict_proba(Z.astype(np.float32)); full=np.zeros((len(Z),S["ncls"]),np.float32)
            for j,c in enumerate(cl): full[:,c]=Pr[:,j]
            return full
    pN=lambda Z: proba(Z)[:,NORMAL]; pred=lambda Z: proba(Z).argmax(1)
    L1=lambda Z,X0,mut:(np.abs(Z[:,mut]-X0[:len(Z),mut])/grng[mut]).sum(1)
    L0=lambda Z,X0,mut:(np.abs(Z[:,mut]-X0[:len(Z),mut])>1e-4).sum(1)
    return pN,pred,L1,L0
def bounds(S,c,sd_k=None):
    sd_k=HARD_SD_K if sd_k is None else sd_k
    le,Xtr,ytr,feat,difs,fidx,gmin,gmax=S["le"],S["Xtr"],S["ytr"],S["feat"],S["difs"],S["fidx"],S["gmin"],S["gmax"]
    ci=le.transform([c])[0]; mm=ytr==ci; mu=Xtr[mm].mean(0); sd=Xtr[mm].std(0)
    lo=(mu-sd_k*sd).astype(np.float32); hi=(mu+sd_k*sd).astype(np.float32)
    mut=[fidx[f] for f in feature_pool(feat,difs) if f not in IMMUTABLE and sd[fidx[f]]>=0.05]
    for f in craft:
        if f in fidx:
            j=fidx[f]; mut.append(j) if j not in mut else None; lo[j]=gmin[j]; hi[j]=gmax[j]
    return ci,mut,lo,hi


In [6]:
# ===== CELL 5 — PHASE 2: generate candidates (cached, incremental) =====
def phase2(gens=None, force=False):
    gens=gens or GENERATORS
    if force and has("p2"): os.remove(f"{CACHE}/p2.pkl")
    S=phase1(); pN,pred,L1,L0=helpers(S); oracle=S["oracle"]
    le,Xte,yte,Xtr,ytr=S["le"],S["Xte"],S["yte"],S["Xtr"],S["ytr"]
    cache=load("p2") if has("p2") else {}; rng=np.random.default_rng(SEED); t=time.time()
    for g in gens:
        if g in HEAVY and not ENABLE_HEAVY: P(f"   {g} skipped (ENABLE_HEAVY=False)"); continue
        gf=GENS[g]
        for c in [x for x in le.classes_ if x!=BEN]:
            if (c,g) in cache: continue
            ci,mut,lo,hi=bounds(S,c); te=Xte[yte==ci]
            if len(te)==0 or not mut: continue
            te=te[rng.choice(len(te),min(EVAL_CAP,len(te)),replace=False)]; det=te[pred(te)==ci]
            if len(det)<10: continue
            tr=Xtr[ytr==ci]; tr=tr[rng.choice(len(tr),min(1200,len(tr)),replace=False)]
            out=(gf(det,tr,lo,hi,mut,pN,L1,rng) if g in NEEDS_TR else gf(det,lo,hi,mut,pN,L1,rng))
            Xa=out[0] if isinstance(out,tuple) else out; q=out[1] if isinstance(out,tuple) else 0
            cmd=cdist(Xa[:,S["cmr"][c]["pidx"]],S["cmr"][c]["centroids"]).min(1)<=S["cmr"][c]["eps"] if c in S["cmr"] else np.ones(len(Xa),bool)
            cache[(c,g)]=dict(ci=ci,mut=mut,det=det,Xa=Xa.astype(np.float32),pN=pN(Xa),Pc=oracle.predict_proba(Xa)[:,ci],
                              l1=L1(Xa,det,mut),l0=L0(Xa,det,mut),cm=cmd,q=q)
        save("p2",cache); P(f"   {g} done t={time.time()-t:.0f}s (cached {len(cache)} blocks)")
    return cache


In [7]:
# ===== CELL 6 — PHASE 3: Tri-Gate (re-runnable) + EXTRACT survivors =====
_THP={}
def _theta(S,ci,mode,fix):
    if mode=="fixed": return fix
    if ci not in _THP: _THP[ci]=max(0.5,float(np.percentile(S["oracle"].predict_proba(S["Xtr"][S["ytr"]==ci][:5000])[:,ci],10)))
    return _THP[ci]
def phase3(theta_mode=None,theta_fix=None,gate2_cmr=None,gate3=None,extract=True,save_name="survivors"):
    theta_mode=THETA_MODE if theta_mode is None else theta_mode
    theta_fix =THETA_FIX  if theta_fix  is None else theta_fix
    gate2_cmr =GATE2_CMR  if gate2_cmr  is None else gate2_cmr
    gate3     =GATE3_OP   if gate3      is None else gate3
    S=phase1(); cache=phase2()
    flds=[f for f in feature_pool(S["feat"],S["difs"]) if f in S["fidx"]]; recs=[]; percl={}
    for (c,g),D in cache.items():
        th=_theta(S,D["ci"],theta_mode,theta_fix); g1=D["pN"]>=GATE1_THR; orc=D["Pc"]>=th
        g2=(orc&D["cm"]) if gate2_cmr else orc; keep=np.where(g1&g2&(D["l1"]<gate3))[0]
        percl[c]=percl.get(c,0)+len(keep)
        if extract:
            for i in keep:
                o=D["Xa"][i]*S["scale_"]+S["mean_"]
                r=dict(cls=c,gen=g,P_Normal=round(float(D["pN"][i]),3),oracle_Pc=round(float(D["Pc"][i]),3),
                       L1=round(float(D["l1"][i]),4),L0=int(D["l0"][i]))
                for f in flds: r[f]=round(float(o[S["fidx"][f]]),4)
                recs.append(r)
    tot=sum(percl.values())
    P(f"[P3] theta={theta_mode}({theta_fix}) Gate2={'oracle&CMR' if gate2_cmr else 'oracle'} L1<{gate3} -> FVAS={tot}  { {k:v for k,v in percl.items() if v>0} }")
    surv=pd.DataFrame(recs)
    if extract and len(surv): surv.to_csv(f"{OUT}/{save_name}.csv",index=False); P(f"     survivors -> {OUT}/{save_name}.csv ({len(surv)} rows)")
    return tot,surv


In [8]:
# ===== CELL 7 — PHASE 4: TABLES + FIGURES (report) =====
# Kaggle-ready: outputs go to /kaggle/working/ + HTML download link

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import os, shutil, glob, base64
from IPython.display import display, HTML

def phase4_report():

    S = phase1(); cache = phase2(); le = S["le"]
    Xtr, ytr, oracle = S["Xtr"], S["ytr"], S["oracle"]
    feat, difs, fidx = S["feat"], S["difs"], S["fidx"]
    mean_, scale_ = S["mean_"], S["scale_"]

    flds = [f for f in feature_pool(feat, difs) if f in fidx]
    classes = sorted({c for (c, _) in cache})
    gens = sorted({g for (_, g) in cache})

    def gt(D, l1t, g2cmr=None, mode=None, fix=None):
        g2cmr = GATE2_CMR if g2cmr is None else g2cmr
        mode = THETA_MODE if mode is None else mode
        fix = THETA_FIX if fix is None else fix
        th = _theta(S, D["ci"], mode, fix)
        g1 = D["pN"] >= GATE1_THR
        orc = D["Pc"] >= th
        return g1, (orc & D["cm"]) if g2cmr else orc, (D["l1"] < l1t)

    # ---------- T1 ----------
    T1 = pd.DataFrame([
        [c, g, len(D["pN"]), round(100 * (D["pN"] >= GATE1_THR).mean(), 1),
         round(100 * (D["Pc"] >= 0.5).mean(), 1), round(float(np.median(D["l1"])), 3),
         round(float(D["l0"].mean()), 1), int(D["q"]),
         round(100 * D["cm"].mean(), 1),
         int((lambda a, b, c: (a & b & c).sum())(*gt(D, GATE3_OP)))]
        for (c, g), D in cache.items()
    ], columns=["class", "gen", "n", "ASR_toNormal%", "FVR%", "medL1", "L0",
                "queries", "CMR%", "FVAS@op"]).sort_values(["class", "gen"])
    T1.to_csv(f"{OUT}/T1_perclass_generator.csv", index=False)

    # ---------- T2 ----------
    g2 = []
    for g in gens:
        sub = [D for (c, gg), D in cache.items() if gg == g]
        g2.append([
            g, "measured",
            round(np.mean([D["pN"].mean() for D in sub]) * 100, 1),
            round(float(np.median(np.concatenate([D["l1"] for D in sub]))), 3),
            round(np.mean([D["l0"].mean() for D in sub]), 1),
            int(np.mean([D["q"] for D in sub])),
            sum(int((lambda a, b, c: (a & b & c).sum())(*gt(D, GATE3_OP))) for D in sub)
        ])
    T2 = pd.DataFrame(g2, columns=["generator", "type", "ASR_toNormal%", "medL1",
                                   "L0", "queries", "FVAS@op"])
    T2.to_csv(f"{OUT}/T2_generator_comparison.csv", index=False)

    # ---------- T3 ----------
    T3 = pd.DataFrame([
        [l,
         sum(int((lambda a, b, c: (a & b & c).sum())(*gt(D, l))) for D in cache.values()),
         sum(int((lambda a, b, c: (a & b & c).sum())(*gt(D, l, True, "paper", 0.5))) for D in cache.values())]
        for l in G3_SWEEP
    ], columns=["L1_budget", "FVAS_modified", "FVAS_strict"])
    T3.to_csv(f"{OUT}/T3_reachability.csv", index=False)

    # ---------- T4 ----------
    T4 = pd.DataFrame([
        [c,
         sum(len(D["pN"]) for (cc, g), D in cache.items() if cc == c),
         sum(int((D["pN"] >= GATE1_THR).sum()) for (cc, g), D in cache.items() if cc == c),
         sum(int(((D["pN"] >= GATE1_THR) & (D["Pc"] >= 0.5)).sum()) for (cc, g), D in cache.items() if cc == c),
         sum(int(((D["pN"] >= GATE1_THR) & (D["Pc"] >= 0.5) & D["cm"]).sum()) for (cc, g), D in cache.items() if cc == c),
         sum(int((lambda a, b, c: (a & b & c).sum())(*gt(D, GATE3_OP))) for (cc, g), D in cache.items() if cc == c)]
        for c in classes
    ], columns=["class", "candidates", "G1", "G1&oracle", "G1&oracle&CMR", "FVAS@op"])
    T4.to_csv(f"{OUT}/T4_funnel.csv", index=False)

    # ---------- Survivors ----------
    BUD = max([l for l in G3_SWEEP if sum(int((lambda a, b, c: (a & b & c).sum())(*gt(D, l))) for D in cache.values()) >= 20] or [G3_SWEEP[-1]])
    SV = []; SM = []
    for (c, g), D in cache.items():
        a, b, cc = gt(D, BUD)
        for i in np.where(a & b & cc)[0]:
            o = D["Xa"][i] * scale_ + mean_
            moved = np.zeros(len(feat), bool)
            for j in D["mut"]:
                moved[j] = abs(D["Xa"][i, j] - D["det"][i, j]) / S["grng"][j] > 1e-4
            r = dict(cls=c, gen=g, L1=round(float(D["l1"][i]), 4), L0=int(D["l0"][i]))
            for f in flds:
                r[f] = round(float(o[fidx[f]]), 4)
            SV.append(r); SM.append(moved)

    survdf = pd.DataFrame(SV)
    SMa = np.array(SM) if SM else np.zeros((0, len(feat)), bool)
    survdf.to_csv(f"{OUT}/survivors_budget_{BUD}.csv", index=False)

    # ---------- T5 ----------
    t5 = []; A6 = []
    for c in (survdf.cls.unique() if len(survdf) else []):
        ic = np.where(survdf.cls.values == c)[0]
        if len(ic) < 8:
            continue
        ci = le.transform([c])[0]
        raw = Xtr[ytr == ci]
        sw = []; rw = []; rh = []
        for f in flds:
            j = fidx[f]
            act = ic[SMa[ic, j]]
            if len(act) < 8:
                continue
            v = survdf.iloc[act][f].values
            slo, shi = np.percentile(v, [2.5, 97.5])
            r = raw[:, j] * scale_[j] + mean_[j]
            rlo, rhi = np.percentile(r, [0.5, 99.5])
            sW = max(shi - slo, 1e-9)
            rW = max(rhi - rlo, 1e-9)
            rho = sW / rW
            sw.append(sW); rw.append(rW); rh.append(rho)
            A6.append([c, f, round(rlo, 3), round(rhi, 3), round(slo, 3), round(shi, 3), round(rho, 3), round(rW / sW, 2)])
        if rh:
            try:
                _, pw = wilcoxon(sw, rw, alternative="less")
            except Exception:
                pw = np.nan
            t5.append([
                c, len(ic), len(rh), round(float(np.median(rh)), 3),
                round(float(np.mean(np.array(rh) < 0.5)), 2),
                round(pw, 4) if pw == pw else None
            ])

    T5 = pd.DataFrame(t5, columns=["class", "n_surv", "n_feat", "median_rho",
                                     "frac_rho<0.5", "Wilcoxon_p"])
    T5.to_csv(f"{OUT}/T5_tightening.csv", index=False)

    A6df = pd.DataFrame(A6, columns=["class", "feature", "raw_lo", "raw_hi",
                                      "surv_lo", "surv_hi", "rho", "tightening_x"])
    A6df.to_csv(f"{OUT}/A6_ranges.csv", index=False)

    # LaTeX tables
    for df, nm, cap in [
        (T2, "T2_generator_comparison", "Generator comparison"),
        (T3, "T3_reachability", "Gate-3 reachability"),
        (T5, "T5_tightening", "Range tightening")
    ]:
        open(f"{OUT}/{nm}.tex", "w").write(
            df.to_latex(index=False, caption=f"{cap} ({DATASET}).", label=f"tab:{nm}")
        )

    # ============================================================
    # FIGURES  —  PNG + PDF + TikZ
    # ============================================================
    figs = []

    def save_all(fig, name, tikz_code=None):
        """Save figure as PNG, individual PDF, and optional TikZ .tex"""
        fig.savefig(f"{OUT}/{name}.png", dpi=120, bbox_inches="tight")
        fig.savefig(f"{OUT}/{name}.pdf", format="pdf", bbox_inches="tight")
        figs.append(fig)
        if tikz_code:
            with open(f"{OUT}/{name}.tex", "w") as f:
                f.write(tikz_code)
        P(f"  -> {name}.png | {name}.pdf" + (f" | {name}.tex" if tikz_code else ""))

    # ---------- F1: FVAS per class ----------
    fv = T1.groupby("class")["FVAS@op"].sum().sort_values(ascending=False)
    f1 = plt.figure(figsize=(9, 4))
    a = f1.gca()
    a.bar(fv.index, fv.values, color="#c0392b")
    a.set_title(f"FVAS/class @L1<{GATE3_OP} ({DATASET},{S['kind']})")
    plt.xticks(rotation=40, ha="right")

    tikz_f1 = r"""\documentclass[tikz,border=5pt]{standalone}
\usepackage{pgfplots}
\pgfplotsset{compat=1.18}
\definecolor{c39red}{RGB}{192,57,43}
\begin{document}
\begin{tikzpicture}
\begin{axis}[
    ybar, width=12cm, height=6cm,
    bar width=8pt,
    xlabel={Class}, ylabel={FVAS},
    title={FVAS per class @L1<""" + str(GATE3_OP) + r""" (""" + DATASET + r""")},
    symbolic x coords={""" + ",".join(fv.index) + r"""},
    xtick=data, x tick label style={rotate=40, anchor=east, font=\small},
    ymin=0, enlarge x limits=0.08,
    legend style={at={(0.98,0.95)},anchor=north east}
]
\addplot[fill=c39red, draw=c39red] coordinates {""" + " ".join([f"({c},{v})" for c, v in fv.items()]) + r"""};
\end{axis}
\end{tikzpicture}
\end{document}"""
    save_all(f1, "F1_fvas", tikz_f1)

    # ---------- F2: Reachability ----------
    f2 = plt.figure(figsize=(7, 4))
    a = f2.gca()
    a.plot(T3.L1_budget, T3.FVAS_modified, "o-", label="modified")
    a.plot(T3.L1_budget, T3.FVAS_strict, "s--", label="strict")
    a.axvline(GATE3_OP, ls=":", color="grey")
    a.set_xlabel("L1 budget")
    a.set_ylabel("FVAS")
    a.set_title("Gate-3 reachability")
    a.legend()

    tikz_f2 = r"""\documentclass[tikz,border=5pt]{standalone}
\usepackage{pgfplots}
\pgfplotsset{compat=1.18}
\begin{document}
\begin{tikzpicture}
\begin{axis}[
    width=10cm, height=6cm,
    xlabel={L1 budget}, ylabel={FVAS},
    title={Gate-3 reachability},
    legend pos=north west,
    grid=both, grid style={line width=.1pt, draw=gray!10},
]
\addplot[mark=*, color=blue] coordinates {""" + " ".join([f"({x},{y})" for x, y in zip(T3.L1_budget, T3.FVAS_modified)]) + r"""};
\addlegendentry{modified}
\addplot[mark=square*, dashed, color=red] coordinates {""" + " ".join([f"({x},{y})" for x, y in zip(T3.L1_budget, T3.FVAS_strict)]) + r"""};
\addlegendentry{strict}
\draw[dotted, thick, gray] (axis cs:""" + str(GATE3_OP) + r""",0) -- (axis cs:""" + str(GATE3_OP) + r""",100);
\end{axis}
\end{tikzpicture}
\end{document}"""
    save_all(f2, "F2_reachability", tikz_f2)

    # ---------- F3: Generators ----------
    mm = T2
    x = np.arange(len(mm))
    f3 = plt.figure(figsize=(8, 4))
    a = f3.gca()
    a.bar(x - 0.2, mm["ASR_toNormal%"], 0.4, label="ASR%")
    a.bar(x + 0.2, mm["FVAS@op"], 0.4, label="FVAS@op")
    a.set_xticks(x)
    a.set_xticklabels(mm.generator)
    a.set_title("Generators")
    a.legend()

    tikz_f3 = r"""\documentclass[tikz,border=5pt]{standalone}
\usepackage{pgfplots}
\pgfplotsset{compat=1.18}
\begin{document}
\begin{tikzpicture}
\begin{axis}[
    ybar, width=11cm, height=6cm,
    bar width=12pt,
    xlabel={Generator}, ylabel={Percentage / Count},
    title={Generators},
    symbolic x coords={""" + ",".join(mm.generator) + r"""},
    xtick=data,
    enlarge x limits=0.15,
    legend pos=north west,
    ymin=0
]
\addplot[fill=blue!60] coordinates {""" + " ".join([f"({g},{v})" for g, v in zip(mm.generator, mm["ASR_toNormal%"])]) + r"""};
\addlegendentry{ASR\%}
\addplot[fill=red!60] coordinates {""" + " ".join([f"({g},{v})" for g, v in zip(mm.generator, mm["FVAS@op"])]) + r"""};
\addlegendentry{FVAS@op}
\end{axis}
\end{tikzpicture}
\end{document}"""
    save_all(f3, "F3_generators", tikz_f3)

    # ---------- F4: Tri-Gate funnel ----------
    f4 = plt.figure(figsize=(10, 4))
    a = f4.gca()
    T4.set_index("class")[["candidates", "G1", "G1&oracle", "G1&oracle&CMR", "FVAS@op"]].plot.bar(ax=a, width=0.85)
    a.set_title("Tri-Gate funnel")
    a.legend(fontsize=7)
    plt.xticks(rotation=40, ha="right")

    tikz_f4 = r"""\documentclass[tikz,border=5pt]{standalone}
\usepackage{pgfplots}
\pgfplotsset{compat=1.18}
\begin{document}
\begin{tikzpicture}
\begin{axis}[
    ybar, width=14cm, height=6cm,
    bar width=6pt,
    xlabel={Class}, ylabel={Count},
    title={Tri-Gate funnel},
    symbolic x coords={""" + ",".join(T4["class"].values) + r"""},
    xtick=data, x tick label style={rotate=40, anchor=east, font=\small},
    enlarge x limits=0.08,
    legend style={at={(0.98,0.95)},anchor=north east,font=\tiny},
    ymin=0
]
\addplot[fill=gray!40] coordinates {""" + " ".join([f"({c},{v})" for c, v in zip(T4["class"], T4["candidates"])]) + r"""};
\addlegendentry{candidates}
\addplot[fill=blue!50] coordinates {""" + " ".join([f"({c},{v})" for c, v in zip(T4["class"], T4["G1"])]) + r"""};
\addlegendentry{G1}
\addplot[fill=green!50] coordinates {""" + " ".join([f"({c},{v})" for c, v in zip(T4["class"], T4["G1&oracle"])]) + r"""};
\addlegendentry{G1\&oracle}
\addplot[fill=orange!60] coordinates {""" + " ".join([f"({c},{v})" for c, v in zip(T4["class"], T4["G1&oracle&CMR"])]) + r"""};
\addlegendentry{G1\&oracle\&CMR}
\addplot[fill=red!70] coordinates {""" + " ".join([f"({c},{v})" for c, v in zip(T4["class"], T4["FVAS@op"])]) + r"""};
\addlegendentry{FVAS@op}
\end{axis}
\end{tikzpicture}
\end{document}"""
    save_all(f4, "F4_funnel", tikz_f4)

    # ---------- F5: Tightening rho ----------
    if len(A6df):
        f5 = plt.figure(figsize=(9, 4))
        a = f5.gca()
        grp = [A6df[A6df["class"] == c]["rho"].values for c in A6df["class"].unique()]
        a.boxplot(grp, labels=list(A6df["class"].unique()))
        a.axhline(0.5, ls="--", color="red")
        a.set_title("Tightening rho")
        plt.xticks(rotation=40, ha="right")

        tikz_f5 = r"""\documentclass[tikz,border=5pt]{standalone}
\usepackage{pgfplots}
\pgfplotsset{compat=1.18}
\begin{document}
\begin{tikzpicture}
\begin{axis}[
    width=12cm, height=6cm,
    xlabel={Class}, ylabel={$\rho$},
    title={Tightening $\rho$},
    symbolic x coords={""" + ",".join(A6df["class"].unique()) + r"""},
    xtick=data, x tick label style={rotate=40, anchor=east, font=\small},
    ymajorgrids=true, grid style={line width=.1pt, draw=gray!10},
    ymin=0,
]
\draw[dashed, thick, red] (axis cs:{""" + A6df["class"].unique()[0] + r"""},0.5) -- (axis cs:{""" + A6df["class"].unique()[-1] + r"""},0.5);
""" + "\n".join([
    r"\addplot+[only marks, mark=*, mark size=1.5pt, opacity=0.6] coordinates {" +
    " ".join([f"({c},{v})" for v in A6df[A6df["class"] == c]["rho"].values]) +
    r"};"
    for c in A6df["class"].unique()
]) + r"""
\end{axis}
\end{tikzpicture}
\end{document}"""
        save_all(f5, "F5_tightening", tikz_f5)

    # ---------- Combined PDF ----------
    with PdfPages(f"{OUT}/CBAG_{DATASET}_figures.pdf") as pp:
        for fg in figs:
            pp.savefig(fg)

    plt.close("all")

    P(f"\n[P4] tables T1-T5 + LaTeX + {len(figs)} figures (PNG+PDF+TikZ) -> {OUT}/")
    P(f"    tightening budget L1<{BUD}")
    P("\nReachability:")
    P(T3.to_string(index=False))
    P("\nGenerators:")
    P(T2.to_string(index=False))

    # ============================================================
    # ZIP & DOWNLOAD — Kaggle compatible (no google.colab)
    # ============================================================
    zip_path = f"{OUT}.zip"
    if os.path.exists(zip_path):
        os.remove(zip_path)

    shutil.make_archive(OUT, 'zip', OUT)
    file_count = len(glob.glob(os.path.join(OUT, '*')))
    zip_size_mb = os.path.getsize(zip_path) / 1024 / 1024
    P(f"\n[ZIP] {file_count} items packed -> {zip_path} ({zip_size_mb:.2f} MB)")

    # Universal HTML base64 download (works in Kaggle, Colab, Jupyter)
    with open(zip_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()

    display(HTML(f"""
    <a href="data:application/zip;base64,{b64}"
       download="CBAG_{DATASET}_results.zip"
       style="padding:12px 24px;background:#2ecc71;color:white;
              text-decoration:none;border-radius:6px;font-family:sans-serif;
              font-weight:bold;display:inline-block;margin-top:10px;
              box-shadow:0 2px 5px rgba(0,0,0,0.2);">
       ⬇️ Download Results ZIP ({zip_size_mb:.2f} MB)
    </a>
    <p style="font-family:sans-serif;color:#666;font-size:12px;margin-top:8px;">
       Or find all files in the <b>Output</b> tab on the right panel.
    </p>
    """))

    P("[DOWNLOAD] Click the green button above, or check the Output tab.")

    return dict(T1=T1, T2=T2, T3=T3, T4=T4, T5=T5)

In [ ]:
# ===== CELL 8 — RUN (edit as needed) =====
# Phase-by-phase; each caches. Re-run any later cell WITHOUT redoing earlier ones.
phase0()                       # load & clean



In [ ]:
phase1()                       # target + oracle + CMR


In [ ]:
# ===== generate in SMALL batches — each caches to disk; if the kernel dies, just re-run =====
phase2(["C-SA","C-PSO"])         # run this cell
phase2(["C-GA","C-RL"])          # then re-run after it finishes (or in a new cell)
phase2(["C-BO","CG-Diffusion","CC-CGAN"])
phase2(["HopSkipJump","Sign-OPT"])
# Tip: if a batch kills the kernel, lower EVAL_CAP (CELL 1) or run ONE generator at a time,
#      e.g. phase2(["C-SA"]).  phase0/phase1 are cached, so restarts resume instantly.

In [ ]:
# Tri-Gate at the paper operating point (targeted):
phase3(extract=True, save_name="survivors_operating")



In [ ]:
# EXTRACT survivors at ANY gate config you want (no regeneration):
# phase3(theta_mode="fixed", theta_fix=0.5, gate2_cmr=False, gate3=0.5, save_name="survivors_L1_0.5")

# Full report (tables + figures + LaTeX):
phase4_report()